In [ ]:
import wave 
import json 
import librosa 
import numpy as np 
from vosk import KaldiRecognizer, Model

In [ ]:
model_path = "vosk-model-small-en-us-0.15"
model = Model(model_path)

In [ ]:
audio_file , sampling_rate = librosa.load("audio_files/03-01-01-01-02-01-02.wav", sr= 16000, mono =True)
audio_data_raw = (audio_file * 32767).astype(np.int16).tobytes()

In [ ]:
recognizer = KaldiRecognizer(model, sampling_rate)

In [ ]:
recognizer.SetWords(True)

In [ ]:
full_transcription = ""
print("Starting transcription...")
recognizer.AcceptWaveform(audio_data_raw)


In [ ]:
final_result = json.loads(recognizer.FinalResult())
full_transcription = final_result.get("text","")
full_transcription

In [ ]:
from IPython.display import Audio, display
display(Audio('audio_files/03-01-01-01-02-01-02.wav'))

In [ ]:
noisy_audio,sampling_rate = librosa.load("audio_files/micrecording_output.wav", sr= 16000, mono = True)
raw_noisy_audio = (noisy_audio*32767).astype(np.int16).tobytes()

In [ ]:
display(Audio('audio_files/micrecording_output.wav'))

In [ ]:
recognizer.AcceptWaveform(raw_noisy_audio)
final_result = json.loads(recognizer.FinalResult())
full_transcript = final_result.get('text','')
full_transcript

In [ ]:
def audio_test(audio_path):
    noisy_audio,sampling_rate = librosa.load(audio_path, sr= 16000, mono = True)
    raw_noisy_audio = (noisy_audio*32767).astype(np.int16).tobytes()
    recognizer.AcceptWaveform(raw_noisy_audio)
    final_result = json.loads(recognizer.FinalResult())
    full_transcript = final_result.get("text","")
    return full_transcript
    

In [ ]:
# no preprocessing 
print(audio_test('audio_files/Standard recording 3.wav'))

In [ ]:
# no preprocessing 
print(audio_test('audio_files/Standard recording 4.wav'))

In [ ]:
print(audio_test('audio_files/noise_reduction_output.wav'))

In [ ]:
print(audio_test('removing silence/cleaned2.wav'))

In [ ]:
# with preprocessing 
print(audio_test('audio_files/Standardrecording3_prepreocessed.wav'))

In [ ]:
display(Audio('audio_files/Standardrecording3_prepreocessed.wav'))

In [ ]:
from scipy.signal import medfilt
import soundfile as sf 

def denoise(audio_path ,new_audio_path):
    noisy_audio,sampling_rate = librosa.load(audio_path, sr= 16000, mono = True)
    s_full , phase = librosa.magphase(librosa.stft(noisy_audio))
    noise_power = np.mean(s_full[:, :int(sampling_rate*0.1) ], axis=1) 
    mask = s_full > noise_power[:, None]
    mask = mask.astype(float)
    mask = medfilt(mask, kernel_size=(1,5))
    s_clean= s_full * mask
    y_clean = librosa.istft(s_clean*phase)
    mask = mask.astype(float)
    mask = medfilt(mask, kernel_size=(1,5))
    s_clean= s_full * mask
    y_clean = librosa.istft(s_clean*phase)
    sf.write(new_audio_path, y_clean, sampling_rate)



In [ ]:
denoise('audio_files/Standard recording 3.wav','audio_files/Standardrecording3_prepreocessed.wav')

In [ ]:
display(Audio('audio_files/Standardrecording3_prepreocessed.wav'))

In [ ]:
display(Audio('audio_files/Standard recording 3.wav'))

In [ ]:
print(audio_test('audio_files/Standardrecording3_prepreocessed.wav'))

In [ ]:
print(audio_test('audio_files/Standard recording 3.wav'))

In [ ]:
denoise('audio_files/Standard recording 4.wav','audio_files/Standardrecording4_prepreocessed.wav')
print(audio_test('audio_files/Standardrecording4_prepreocessed.wav'))

In [ ]:
print(audio_test('audio_files/Standard recording 4.wav'))

In [ ]:
import webrtcvad
def frame_generator(y_int16, sr, frame_duration=30):
    n = int(sr * frame_duration / 1000)
    for offset in range(0, len(y_int16), n):
        frame = y_int16[offset:offset+n]

        if len(frame) < n:
            pad = np.zeros(n - len(frame), dtype=np.int16)
            frame = np.concatenate([frame, pad])
        yield frame

def vad_filter_safe(y, sr=16000, mode=2, frame_duration=30):
    y = np.clip(y, -1.0, 1.0)
    y_int16 = (y * 32767).astype(np.int16)
    vad = webrtcvad.Vad(mode)
    speech_frames = []

    for i, frame in enumerate(frame_generator(y_int16, sr, frame_duration)):
        if vad.is_speech(frame.tobytes(), sr):
            speech_frames.append(frame)
            
    if not speech_frames:
        return np.array([], dtype=np.float32)
    speech = np.concatenate(speech_frames)
    speech = speech.astype(np.float32) / 32767.0
    return speech

def silence_removal(audio_path ,new_audio_path):
    y_cleaned , sr = librosa.load(audio_path, mono=True)
    y_vad = vad_filter_safe(y_cleaned, sr=16000, mode=2)
    sf.write(new_audio_path, y_vad, sr, subtype='PCM_16')

    

In [ ]:
silence_removal('audio_files/Standard recording 4.wav','audio_files/Standardrecording4_prepreocessed.wav')

In [ ]:
display(Audio('audio_files/Standardrecording4_prepreocessed.wav'))

In [ ]:
print(audio_test('audio_files/Standardrecording4_prepreocessed.wav'))

In [ ]:
print(audio_test('audio_files/Standard recording 4.wav'))

In [ ]:
silence_removal('audio_files/Standard recording 3.wav','audio_files/Standardrecording3_prepreocessed.wav')
display(Audio('audio_files/Standardrecording3_prepreocessed.wav'))

In [ ]:
print(audio_test('audio_files/Standardrecording3_prepreocessed.wav')),
print(audio_test('audio_files/Standard recording 3.wav'))


In [ ]:
print(audio_test('test_audio_files/Standard recording20.wav'))

In [ ]:
print(audio_test('test_audio_processed/Standard recording20.wav'))

In [ ]:
from glob import glob
audio_files = glob('test_audio_files/*.wav')
for i in audio_files:
    print(audio_test(i))

In [ ]:
test_data = glob('archive/Hermana_Molly/Hermana_Molly/*.wav')
for i in test_data:
    print(audio_test(i))
    display(Audio(i))

In [ ]:
denoise('test_audio_files/Standard recording20.wav','test_audio_processed/Standard recording20.wav')
silence_removal('test_audio_processed/Standard recording20.wav', 'test_audio_processed/Standard recording20.wav')
display(Audio('test_audio_processed/Standard recording20.wav'))
print(audio_test('test_audio_processed/Standard recording20.wav'))